# Bronze: Scraping Spesifikasi HP (GSMArena) → DuckDB

Notebook ini **berdiri sendiri** (bisa dijalankan tanpa bergantung notebook lain) dan menghasilkan **satu file
`data/gsmarena.duckdb`** — lapisan *bronze* (data mentah per sumber) dalam arsitektur:

```
gsmarena.duckdb ─┐
tokopedia.duckdb ─┼──► (notebook build_ducklake, menyusul) ──► satu DuckLake (gold) ──► Streamlit
shopee.duckdb    ─┤        HP dari GSMArena dipakai sebagai
youtube.duckdb   ─┘        daftar keyword pencarian di Tokopedia & Shopee
```

## Kenapa DuckDB, dan kenapa path relatif
- **DuckDB** dipakai menggantikan CSV: satu file `.duckdb` berisi tabel `phone_specs`, dan **checkpoint-nya
  adalah tabel itu sendiri** — jalankan ulang sel scraping kapan saja, HP yang sudah tersimpan otomatis dilewati.
- **Path relatif** (`data/gsmarena.duckdb`, relatif terhadap folder notebook ini) supaya siapa pun di kelompok
  yang `git clone` repo dan membuka notebook ini langsung dapat path yang benar, tanpa mengedit apa pun.
- **File `.duckdb` akan di-commit ke GitHub.** GitHub membatasi keras 100 MB/file (memperingatkan di atas 50 MB).
  Notebook ini mencetak ukuran file setiap kali selesai supaya Anda tahu sebelum `git push`.

## DuckDB tetap sumber utama, CSV untuk lampiran laporan
Checkpoint & semua logika (resume, dibaca notebook lain) tetap berbasis **DuckDB** (`phone_specs`). Setiap kali
scraping selesai (atau berhenti karena diblokir/error), tabel itu **diekspor ulang ke `data/raw_specs.csv`**
supaya ada file yang mudah dibuka di Excel/dilampirkan ke laporan. CSV ini murni salinan untuk dibaca manusia —
notebook Tokopedia/Shopee/build_ducklake tetap membaca dari `.duckdb`, bukan dari CSV ini.

## Kenapa iPad/Watch/Tab ikut kesaring, dan cara mengatasinya
Halaman daftar brand di GSMArena (`apple-phones-48.php`, `samsung-phones-9.php`, dsb.) **memang mencampur**
HP dengan tablet & smartwatch brand yang sama — ini perilaku asli GSMArena, bukan bug scraping sebelumnya.
Notebook ini menyaring dua lapis: (1) berdasarkan **URL** sebelum halaman detailnya dibuka (hemat request), dan
(2) berdasarkan **nama produk** setelah didapat, sebagai jaring pengaman. Kata kunci non-HP: `ipad`, `watch`,
`tab`, `pad`, `band`, `buds`, `book`, `ring`, `glass`, `tv`, `chromebook`, `tablet`, dsb. — lihat `NON_PHONE_TOKENS`.

## Isi tabel `phone_specs`
Kolom **mentah** (belum dibersihkan) plus `release_year` yang diparse untuk keperluan logika berhenti otomatis
(supaya scraping tidak melebar ke HP terlalu lama). Pembersihan menyeluruh (pisah chipset, parsing RAM/storage,
dsb.) dilakukan nanti di notebook `build_ducklake`, bukan di sini — prinsip lapisan bronze: simpan apa adanya.

> ⚠️ Catatan yang sama seperti sebelumnya masih berlaku: GSMArena bisa timeout/DNS error (masalah jaringan lokal)
> atau membatasi laju request (429 / koneksi ditutup paksa) jika terlalu banyak/cepat. Lihat bagian 4 untuk detail
> penanganannya.

## 1. Instalasi & Import Library

In [ ]:
%pip install -q requests beautifulsoup4 lxml pandas duckdb

In [11]:
import re
import os
import time
import random
from pathlib import Path
from datetime import datetime
from urllib.parse import urljoin

import requests
import pandas as pd
import duckdb
from bs4 import BeautifulSoup

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 2. Konfigurasi

In [ ]:
# Path RELATIF terhadap folder tempat notebook ini berada (bukan folder home) — supaya sama untuk semua anggota
# kelompok setelah `git clone`. Jalankan notebook ini dari dalam repo (folder repo = working directory).
DATA_DIR = Path("data")
DB_DIR = Path("database")
DATA_DIR.mkdir(exist_ok=True)
DB_DIR.mkdir(exist_ok=True)
DB_PATH = DB_DIR / "gsmarena.duckdb"
CSV_PATH = DATA_DIR / "raw_dataset_gsmarena.csv"          # salinan untuk lampiran laporan (bukan sumber data)
print("Database bronze GSMArena:", DB_PATH.resolve())
print("Salinan CSV untuk laporan:", CSV_PATH.resolve())

GITHUB_WARN_MB = 50    # GitHub mulai memperingatkan di atas ukuran ini; batas keras GitHub 100 MB/file

# Brand diurutkan brand populer Indonesia lebih dulu (supaya kalau proses terhenti di tengah / kena rate-limit,
# brand yang paling penting untuk analisis sudah lebih dulu selesai)
TARGET_BRANDS = ["Samsung", "Xiaomi", "Oppo", "Vivo", "Realme", "Infinix", "Tecno", "itel",
                 "Apple", "Huawei"]
MAX_PHONES_PER_BRAND = 30       # batas HP per brand yang masuk jendela tahun (None = tanpa batas)
MAX_LINK_PAGES_PER_BRAND = 10    # batas halaman daftar yang dibaca per brand (dari terbaru ke lama)
MIN_YEAR = 2025                 # hanya HP rilis >= tahun ini (2024, 2025, 2026 = ± 2,7 tahun terakhir)
OLD_STREAK_STOP = 4              # berhenti membaca brand jika 4 HP berturut-turut sudah lebih lama dari MIN_YEAR
GSM_DELAY = (2.5, 5.0)           # jeda acak antar request (detik) — jangan diperkecil, GSMArena membatasi laju request

# Kata (token utuh, bukan substring) yang menandakan produk BUKAN HP — dicek dari slug URL dan nama produk
NON_PHONE_TOKENS = {"ipad", "watch", "tab", "pad", "band", "buds", "earbuds", "book", "ring",
                    "glass", "tv", "chromebook", "monitor", "speaker", "router", "tablet"}

Database bronze GSMArena: C:\KULIAH\Semester 7\Data Mining II\Tugas\PJBL\data\gsmarena.duckdb
Salinan CSV untuk laporan: C:\KULIAH\Semester 7\Data Mining II\Tugas\PJBL\data\raw_specs.csv


## 3. Koneksi ke `data/gsmarena.duckdb`
Tabel `phone_specs` dibuat jika belum ada. Koneksi dibuka **sekali** dan dipakai sepanjang notebook — jangan
buka notebook ini dua kali sekaligus (DuckDB mengunci file saat dipakai; jika muncul error "file is already
open/locked", restart kernel di jendela yang lain).

In [13]:
con = duckdb.connect(str(DB_PATH))
con.execute("""CREATE TABLE IF NOT EXISTS phone_specs (
    brand VARCHAR, phone_name VARCHAR, chipset_raw VARCHAR, internalmemory_raw VARCHAR,
    battery_raw VARCHAR, released_raw VARCHAR, release_year INTEGER,
    gsmarena_url VARCHAR PRIMARY KEY, scraped_at TIMESTAMP
)""")
n_existing = con.execute("SELECT count(*) FROM phone_specs").fetchone()[0]
print(f"Tabel phone_specs siap. Baris yang sudah ada (checkpoint): {n_existing}")


def export_csv():
    # Menyalin ISI TABEL SAAT INI (bukan append baris manual) ke CSV, supaya CSV selalu persis sama dengan
    # DuckDB — aman dipanggil berkali-kali, tidak akan menghasilkan baris dobel meski dipanggil berulang.
    df = con.execute("SELECT * FROM phone_specs ORDER BY brand, scraped_at").df()
    df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
    print(f"✔ {CSV_PATH} diperbarui ({len(df)} baris)")

Tabel phone_specs siap. Baris yang sudah ada (checkpoint): 200


## 4. Extract: GSMArena (Requests + BeautifulSoup)
Sama seperti sebelumnya: tiap brand dibaca dari **terbaru ke lama**, berhenti otomatis setelah `OLD_STREAK_STOP`
HP berturut-turut lebih lama dari `MIN_YEAR`, atau setelah `MAX_PHONES_PER_BRAND` tercapai. Bedanya, checkpoint
sekarang langsung berupa baris di tabel `phone_specs` (bukan file CSV terpisah).

In [14]:
GSM_HOME = "https://www.gsmarena.com/"
FALLBACK_BRAND_URLS = {
    "Samsung": GSM_HOME + "samsung-phones-9.php", "Xiaomi": GSM_HOME + "xiaomi-phones-80.php",
    "Oppo": GSM_HOME + "oppo-phones-82.php", "Vivo": GSM_HOME + "vivo-phones-98.php",
    "Realme": GSM_HOME + "realme-phones-118.php", "Apple": GSM_HOME + "apple-phones-48.php",
}

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
})


def polite_sleep(a=2.5, b=5.0):
    time.sleep(random.uniform(a, b))


def fetch_soup(url, retries=3):
    for attempt in range(retries):
        try:
            resp = SESSION.get(url, timeout=(10, 45))
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "lxml")
            if resp.status_code == 429:
                wait = 30 * (attempt + 1)
                print(f"   429 Too Many Requests, tunggu {wait} detik...")
                time.sleep(wait)
            else:
                print(f"   HTTP {resp.status_code} untuk {url}")
                time.sleep(5)
        except requests.exceptions.ConnectionError as e:
            # DNS gagal / koneksi diputus paksa: masalah jaringan, bukan pemblokiran — tunggu lalu coba lagi
            print(f"   Masalah jaringan ({type(e).__name__}), tunggu 15 detik lalu coba lagi...")
            time.sleep(15)
        except requests.RequestException as e:
            print(f"   Error koneksi: {e}")
            time.sleep(5)
    return None


def discover_brands(target_names):
    wanted = {n.lower(): n for n in target_names}
    found = {}
    soup = fetch_soup(urljoin(GSM_HOME, "makers.php3"))
    if soup is not None:
        for a in soup.select("div.st-text table a"):
            first = a.find(string=True)
            name = first.strip() if first else ""
            if name.lower() in wanted:
                found[wanted[name.lower()]] = urljoin(GSM_HOME, a["href"])
    if not found:
        print("⚠️ Penemuan brand otomatis gagal, memakai daftar URL cadangan.")
        found = {k: v for k, v in FALLBACK_BRAND_URLS.items() if k in target_names}
    missing = [n for n in target_names if n not in found]
    if missing:
        print("Brand tidak ditemukan di GSMArena (dilewati):", missing)
    return found


def iter_brand_links(brand_url, max_pages):
    # Menghasilkan link HP halaman demi halaman (terbaru -> lama)
    url, page, seen = brand_url, 0, set()
    while url and page < max_pages:
        soup = fetch_soup(url)
        if soup is None:
            return
        for a in soup.select("div.makers ul li a"):
            link = urljoin(GSM_HOME, a["href"])
            if link in seen:
                continue
            seen.add(link)
            if not looks_like_phone_url(link):
                continue   # kemungkinan besar iPad/Watch/Tab dsb. — dilewati tanpa membuka halaman detailnya
            yield link
        nxt = soup.select_one("a.pages-next")
        href = nxt.get("href") if nxt else None
        url = urljoin(GSM_HOME, href) if href and not href.startswith("#") else None
        page += 1
        polite_sleep(*GSM_DELAY)


def parse_release_year(text):
    m = re.search(r"\b((?:19|20)\d{2})\b", text or "")
    return int(m.group(1)) if m else None


def looks_like_phone_url(url):
    # Cek dari slug URL (mis. "samsung_galaxy_tab_s10-13361.php" -> token {samsung, galaxy, tab, s10})
    # SEBELUM halaman detail dibuka, supaya tablet/watch tidak perlu di-request sama sekali (hemat kuota request)
    slug = re.sub(r"-\d+\.php$", "", url.rsplit("/", 1)[-1])
    tokens = set(slug.lower().split("_"))
    return not (tokens & NON_PHONE_TOKENS)


def looks_like_phone_name(name):
    # Jaring pengaman kedua, dari nama produk hasil parsing (untuk kasus slug URL tidak jelas)
    tokens = set(re.sub(r"[^\w\s]", " ", (name or "").lower()).split())
    return not (tokens & NON_PHONE_TOKENS)


def parse_phone_page(url, brand):
    soup = fetch_soup(url)
    if soup is None:
        return None
    title = soup.select_one("h1.specs-phone-name-title")
    if title is None:
        return None

    def spec(key):
        el = soup.select_one(f'[data-spec="{key}"]')
        return el.get_text(" ", strip=True) if el else None

    released_raw = spec("year")
    return {
        "brand": brand, "phone_name": title.get_text(strip=True), "chipset_raw": spec("chipset"),
        "internalmemory_raw": spec("internalmemory"), "battery_raw": spec("batdescription1"),
        "released_raw": released_raw, "release_year": parse_release_year(released_raw), "gsmarena_url": url,
    }

In [5]:
def scrape_gsmarena():
    year_of = dict(con.execute("SELECT gsmarena_url, release_year FROM phone_specs").fetchall())
    if year_of:
        print(f"Checkpoint (dari database): {len(year_of)} HP sudah pernah diambil, dilewati.\n")

    brands = discover_brands(TARGET_BRANDS)
    fail_streak = 0
    for brand, brand_url in brands.items():
        taken = old_streak = 0
        print(f"[GSMArena] {brand}")
        for link in iter_brand_links(brand_url, MAX_LINK_PAGES_PER_BRAND):
            if link in year_of:
                year = year_of[link]
            else:
                polite_sleep(*GSM_DELAY)
                data = parse_phone_page(link, brand)
                if data is None:
                    fail_streak += 1
                    if fail_streak >= 10:
                        print("⛔ 10 kegagalan berturut-turut (kemungkinan diblokir sementara, atau jaringan "
                              "tidak stabil). Berhenti dulu, tunggu beberapa menit / ganti jaringan lalu "
                              "jalankan ulang sel ini untuk melanjutkan (checkpoint sudah tersimpan di database).")
                        export_csv()
                        return
                    continue
                fail_streak = 0
                if not looks_like_phone_name(data["phone_name"]):
                    print(f"   (dilewati, sepertinya bukan HP: {data['phone_name']})")
                    continue   # tidak dimasukkan ke year_of -> tidak ikut dihitung old_streak/taken
                con.execute("INSERT OR IGNORE INTO phone_specs VALUES (?,?,?,?,?,?,?,?, now())", [
                    data["brand"], data["phone_name"], data["chipset_raw"], data["internalmemory_raw"],
                    data["battery_raw"], data["released_raw"], data["release_year"], data["gsmarena_url"]])
                year = data["release_year"]
                year_of[link] = year

            if year is None:                 # batal rilis / rumor: tidak dihitung
                continue
            if MIN_YEAR and year < MIN_YEAR:
                old_streak += 1
                if old_streak >= OLD_STREAK_STOP:
                    print(f"   → {old_streak} HP berturut-turut sebelum {MIN_YEAR}, berhenti untuk brand ini ({taken} HP dalam jendela tahun)")
                    break
            else:
                old_streak = 0
                taken += 1
                if MAX_PHONES_PER_BRAND and taken >= MAX_PHONES_PER_BRAND:
                    print(f"   → batas {MAX_PHONES_PER_BRAND} HP per brand tercapai")
                    break

    export_csv()
    size_mb = DB_PATH.stat().st_size / 1_000_000
    flag = " ⚠️ mendekati/melebihi batas GitHub!" if size_mb > GITHUB_WARN_MB else ""
    print(f"\nUkuran {DB_PATH.name}: {size_mb:.1f} MB{flag}")

## 5. Jalankan Extract
Proses ini bisa memakan waktu (± 1–1,5 jam untuk jendela 2024–2026, 22 brand). Bisa dihentikan & dilanjutkan kapan saja — checkpoint tersimpan langsung di `phone_specs`.

In [17]:
scrape_gsmarena()

n = con.execute("SELECT count(*) FROM phone_specs WHERE release_year >= ?", [MIN_YEAR]).fetchone()[0]
if n == 0:
    raise RuntimeError("Belum ada HP tersimpan. Periksa pesan error di atas (jaringan/pemblokiran), lalu jalankan ulang sel ini.")
print(f"\nTotal HP tersimpan (rilis >= {MIN_YEAR}): {n}")

Checkpoint (dari database): 282 HP sudah pernah diambil, dilewati.

[GSMArena] Apple
   → 4 HP berturut-turut sebelum 2025, berhenti untuk brand ini (9 HP dalam jendela tahun)
[GSMArena] Huawei
   → batas 30 HP per brand tercapai
[GSMArena] Infinix
   → 4 HP berturut-turut sebelum 2025, berhenti untuk brand ini (27 HP dalam jendela tahun)
[GSMArena] itel
   → 4 HP berturut-turut sebelum 2025, berhenti untuk brand ini (24 HP dalam jendela tahun)
[GSMArena] Oppo
   → batas 30 HP per brand tercapai
[GSMArena] Realme
   → batas 30 HP per brand tercapai
[GSMArena] Samsung
   → batas 30 HP per brand tercapai
[GSMArena] Tecno
   → batas 30 HP per brand tercapai
[GSMArena] Vivo
   → batas 30 HP per brand tercapai
[GSMArena] Xiaomi
   → batas 30 HP per brand tercapai
✔ data\raw_specs.csv diperbarui (282 baris)

Ukuran gsmarena.duckdb: 0.0 MB

Total HP tersimpan (rilis >= 2025): 270


## 6. Bersihkan Data yang Sudah Terlanjur Tersimpan (jalankan sekali saja)
Jika Anda sudah pernah menjalankan versi sebelumnya (sebelum ada penyaringan di atas), tabel `phone_specs`
mungkin masih berisi iPad/Watch/Tab dsb. Sel ini membuang baris yang namanya cocok dengan `NON_PHONE_TOKENS`,
tanpa perlu scraping ulang dari awal. **Aman dijalankan berkali-kali** — jika tidak ada yang perlu dibuang, tidak
ada yang terjadi.

In [18]:
all_rows = con.execute("SELECT phone_name, gsmarena_url FROM phone_specs").df()
to_remove = all_rows[~all_rows["phone_name"].apply(looks_like_phone_name)]

if len(to_remove):
    print(f"Membuang {len(to_remove)} baris yang sepertinya bukan HP:")
    print(to_remove["phone_name"].to_string(index=False))
    con.execute("DELETE FROM phone_specs WHERE gsmarena_url IN (SELECT unnest(?))", [to_remove["gsmarena_url"].tolist()])
    export_csv()
else:
    print("Tidak ada baris non-HP yang perlu dibuang.")

print(f"\nTotal baris sekarang: {con.execute('SELECT count(*) FROM phone_specs').fetchone()[0]}")

Tidak ada baris non-HP yang perlu dibuang.

Total baris sekarang: 282


## 7. Pratinjau Data

In [19]:
con.execute("""
    SELECT brand, count(*) AS jumlah_hp, min(release_year) AS tahun_min, max(release_year) AS tahun_maks
    FROM phone_specs WHERE release_year >= ? GROUP BY brand ORDER BY jumlah_hp DESC
""", [MIN_YEAR]).df()

,brand,jumlah_hp,tahun_min,tahun_maks
0,Realme,30,2025,2026
1,Samsung,30,2025,2026
2,Tecno,30,2025,2026
3,Oppo,30,2026,2026
4,Xiaomi,30,2026,2026
5,Huawei,30,2025,2026
6,Vivo,30,2026,2026
7,Infinix,27,2025,2026
8,itel,24,2025,2026
9,Apple,9,2025,2026


In [20]:
con.execute("SELECT * FROM phone_specs ORDER BY scraped_at DESC LIMIT 10").df()

,brand,phone_name,chipset_raw,internalmemory_raw,battery_raw,released_raw,release_year,gsmarena_url,scraped_at
0,Xiaomi,Xiaomi Poco C81x,Unisoc T7250 (12 nm),64GB 3GB RAM,5200 mAh,"2026, April 23",2026,https://www.gsmarena.com/xiaomi_poco_c81x-1462...,2026-09-30 19:19:52.482014
1,Xiaomi,Xiaomi Poco C81,Unisoc T7250 (12 nm),64GB 4GB RAM,6300 mAh,"2026, April 23",2026,https://www.gsmarena.com/xiaomi_poco_c81-14626...,2026-09-30 19:19:47.724872
2,Xiaomi,Xiaomi Poco C81 Pro,Unisoc T7250 (12 nm),"64GB 4GB RAM, 128GB 4GB RAM, 256GB 4GB RAM",6000 mAh,"2026, April 27",2026,https://www.gsmarena.com/xiaomi_poco_c81_pro-1...,2026-09-30 19:19:43.482105
3,Xiaomi,Xiaomi 17 Max,Qualcomm SM8850-AC Snapdragon 8 Elite Gen 5 (3...,"256GB 12GB RAM, 256GB 16GB RAM, 512GB 12GB RAM...",Si/C Li-Ion 8000 mAh,"2026, May 21",2026,https://www.gsmarena.com/xiaomi_17_max_5g-1466...,2026-09-30 19:19:39.659923
4,Xiaomi,Xiaomi 17T,Mediatek Dimensity 8500 Ultra (4 nm),"256GB 12GB RAM, 512GB 12GB RAM",Si/C Li-Ion 6500 mAh - Global Si/C Li-Ion 7000...,"2026, May 28",2026,https://www.gsmarena.com/xiaomi_17t_5g-14652.php,2026-09-30 19:19:35.002002
5,Xiaomi,Xiaomi 17T Pro,Mediatek Dimensity 9500 (3 nm),"256GB 12GB RAM, 512GB 12GB RAM, 512GB 16GB RAM...",Si/C Li-Ion 7000 mAh,"2026, May 28",2026,https://www.gsmarena.com/xiaomi_17t_pro_5g-146...,2026-09-30 19:19:29.782419
6,Xiaomi,Xiaomi Redmi 17C 4G,Mediatek Helio G81 Ultra (12 nm),"64GB 4GB RAM, 128GB 4GB RAM",5160 mAh,"2026, June 23",2026,https://www.gsmarena.com/xiaomi_redmi_17c_(chi...,2026-09-30 19:19:24.550905
7,Xiaomi,Xiaomi Redmi K90 Ultra,Qualcomm SM8750-AB Snapdragon 8 Elite (3 nm),"256GB 12GB RAM, 256GB 16GB RAM, 512GB 12GB RAM...",Si/C Li-Ion 8550 mAh,"2026, June 30",2026,https://www.gsmarena.com/xiaomi_redmi_k90_ultr...,2026-09-30 19:19:18.960202
8,Xiaomi,Xiaomi Redmi Note 17 (India/China),Qualcomm SM4450-AF Snapdragon 4 Gen 4,"128GB 6GB RAM, 128GB 8GB RAM, 256GB 8GB RAM",Si/C Li-Ion 8000 mAh,"2026, July 14",2026,https://www.gsmarena.com/xiaomi_redmi_note_17_...,2026-09-30 19:19:13.181809
9,Xiaomi,Xiaomi Redmi Note 17 Pro (China),Qualcomm SM6435-AA Snapdragon 6s Gen 4 (4 nm),"128GB 8GB RAM, 256GB 8GB RAM, 256GB 12GB RAM, ...",Si/C Li-Ion 9000 mAh,"2026, July 14",2026,https://www.gsmarena.com/xiaomi_redmi_note_17_...,2026-09-30 19:19:08.778771


## Catatan
- **Lapisan bronze, belum dibersihkan.** Kolom `*_raw` masih berupa teks mentah dari GSMArena (mis. `"128GB 8GB
  RAM"`, `"Exynos 1480 (4 nm)"`). Pembersihan (pisah chipset, parsing RAM/storage ke angka, dsb.) dilakukan nanti
  di notebook `build_ducklake`, supaya bronze tetap jadi salinan apa adanya dari sumber aslinya — prinsip umum
  arsitektur medallion (bronze = mentah, silver/gold = sudah dibersihkan & terintegrasi).
- **File ini dipakai notebook Tokopedia & Shopee** sebagai sumber daftar nama HP (keyword pencarian). Pastikan
  `data/gsmarena.duckdb` sudah punya cukup baris sebelum menjalankan notebook itu.
- **`data/raw_specs.csv`** hanya salinan untuk dibaca manusia/dilampirkan ke laporan — selalu dibuat ulang dari isi
  tabel DuckDB (bukan ditulis manual), jadi aman jika sel scraping dijalankan berkali-kali. Kalau butuh salinan
  terbaru di luar proses scraping (mis. setelah menghapus baris tertentu manual di DuckDB), panggil `export_csv()`
  di sel baru kapan saja.
- **Sebelum `git push`**, jalankan `du -h data/gsmarena.duckdb` (atau lihat ukuran di sel bagian 4) untuk memastikan
  belum melewati batas GitHub. Jika mendekati, pertimbangkan Git LFS atau mempersempit `MIN_YEAR`/`MAX_PHONES_PER_BRAND`.
- **Jaringan vs pemblokiran**: error DNS/`ConnectionError` sekarang ditunggu & dicoba lagi otomatis (bukan langsung
  dihitung sebagai kegagalan berat), karena itu biasanya masalah jaringan lokal, bukan GSMArena memblokir. Hanya
  429/kegagalan berulang yang dianggap tanda pembatasan laju request.